# PaperRAG Ingest Notebook

这个 notebook 专门给 Colab 上跑 `ingest --force` 用。

推荐流程：
1. 在 Colab 打开这个 notebook
2. 修改下面的配置 cell，填入 MinerU 和 Milvus 参数
3. 直接使用仓库里的 `data/pdf`，或者按需再额外上传 PDF
4. 运行 `ingest --force`
5. 下载打包好的 `data/cache` 和 `data/mineru_output`

注意：
- 本 notebook 不依赖本地 `.env`
- 默认假设 PDF 已经通过 GitHub 放进仓库的 `data/pdf`
- 只有当你想临时补充 PDF 时，才把 `USE_COLAB_UPLOAD` 改成 `True`
- `ask` 本地直用真正依赖的是 `data/cache` + 同一套云端 Milvus
- `data/mineru_output` 不是问答必须，但建议一起带回本地，后续增量入库可以复用解析结果


In [ ]:
from pathlib import Path

# ===== repo =====
REPO_OWNER = 'YaoHui-Wu06022'
REPO_NAME = 'PaperRAG'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/PaperRAG')

# ===== upload / output =====
# 默认直接使用仓库里 data/pdf 的论文。
# 只有当你想在 Colab 临时补充 PDF 时，才把 USE_COLAB_UPLOAD 改成 True。
UPLOAD_DIR = Path('/content/upload_pdfs')
ARTIFACT_DIR = Path('/content/ingest_artifacts')
CLEAR_REPO_PDFS = False
DOWNLOAD_AFTER_INGEST = True
USE_COLAB_UPLOAD = False

# ===== secrets: fill these before running =====
MINERU_API_TOKEN = 'paste_your_mineru_token'
MILVUS_URI = 'https://your-zilliz-endpoint'
MILVUS_TOKEN = 'paste_your_milvus_token'
MILVUS_DB_NAME = ''
EMBEDDING_API_KEY = 'paste_your_jina_api_key'

# ===== config source =====
# notebook 会先读取仓库里的 .env.example，确保非敏感参数始终跟仓库配置一致。
ENV_TEMPLATE_PATH = Path('.env.example')

# ===== optional notebook-only overrides =====
# 只在你明确想临时覆盖 .env.example 某些值时才改这里。
# 如果这里为空，运行参数就完全跟仓库里的 .env.example 保持一致。
EXTRA_ENV_OVERRIDES = {
    # 'QUERY_REWRITE_MAX_VARIANTS': '2',
}

print({
    'repo': f'{REPO_OWNER}/{REPO_NAME}@{REPO_BRANCH}',
    'upload_dir': str(UPLOAD_DIR),
    'artifact_dir': str(ARTIFACT_DIR),
    'env_template': str(ENV_TEMPLATE_PATH),
    'use_colab_upload': USE_COLAB_UPLOAD,
    'clear_repo_pdfs': CLEAR_REPO_PDFS,
})


In [ ]:
%pip install -qU python-dotenv requests pymilvus langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-openai langchain-milvus sentence-transformers transformers openai pymupdf pillow pandas rank-bm25


In [ ]:
import os
import shutil
import subprocess
import sys

# 克隆公开仓库到 Colab 工作目录。
# 每次运行都重拉一份，避免沿用旧状态。
clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repo_ready=', REPO_DIR)


In [ ]:
from pathlib import Path
from zipfile import ZipFile

# 这一步先检查仓库里自带的 PDF。
# 如果 USE_COLAB_UPLOAD=True，再额外允许你从 Colab 临时上传 PDF 或 ZIP。
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
repo_pdf_dir = REPO_DIR / 'data' / 'pdf'
repo_pdf_dir.mkdir(parents=True, exist_ok=True)

repo_pdf_paths = sorted(repo_pdf_dir.glob('*.pdf'))
print('repo_pdf_count=', len(repo_pdf_paths))
for path in repo_pdf_paths[:20]:
    print('-', path.name)

uploaded_pdf_paths = []

if USE_COLAB_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        target = UPLOAD_DIR / name
        target.write_bytes(data)
        if target.suffix.lower() == '.zip':
            with ZipFile(target, 'r') as zf:
                zf.extractall(UPLOAD_DIR)
    uploaded_pdf_paths = sorted([p for p in UPLOAD_DIR.rglob('*') if p.suffix.lower() == '.pdf'])

print('uploaded_pdf_count=', len(uploaded_pdf_paths))
for path in uploaded_pdf_paths[:20]:
    print('-', path.name)

if not repo_pdf_paths and not uploaded_pdf_paths:
    raise FileNotFoundError('No PDF found in repo data/pdf or uploaded files')


In [ ]:
import shutil

# 如果你启用了 Colab 上传，这一步负责把临时上传的 PDF 同步到仓库目录。
# 默认情况下不会动仓库里的 PDF，直接用 GitHub 上已有的内容入库。
repo_pdf_dir = REPO_DIR / 'data' / 'pdf'
repo_pdf_dir.mkdir(parents=True, exist_ok=True)

if USE_COLAB_UPLOAD and uploaded_pdf_paths:
    if CLEAR_REPO_PDFS:
        for old_pdf in repo_pdf_dir.glob('*.pdf'):
            old_pdf.unlink()
    for pdf in uploaded_pdf_paths:
        shutil.copy2(pdf, repo_pdf_dir / pdf.name)
    print('copied_uploaded_pdfs=', len(uploaded_pdf_paths))
else:
    print('using_repo_pdfs_only=True')

final_pdf_paths = sorted(repo_pdf_dir.glob('*.pdf'))
print('final_repo_pdf_count=', len(final_pdf_paths))
for path in final_pdf_paths[:20]:
    print('-', path.name)


In [ ]:
import json
import os
import torch
from dotenv import dotenv_values

# 先读取仓库里的 .env.example，作为非敏感参数模板；
# 再覆盖 MinerU / Milvus 凭据和 Colab 本地运行相关变量。
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

env_template_path = REPO_DIR / ENV_TEMPLATE_PATH
if not env_template_path.exists():
    raise FileNotFoundError(f'.env.example not found: {env_template_path}')

template_env = {
    str(key): '' if value is None else str(value)
    for key, value in dotenv_values(env_template_path).items()
}
for key, value in template_env.items():
    os.environ[str(key)] = str(value)

# 这些值不写进 .env.example，只在本次 Colab 会话里覆盖。
os.environ['MINERU_API_TOKEN'] = str(MINERU_API_TOKEN).strip()
os.environ['MILVUS_URI'] = str(MILVUS_URI).strip()
os.environ['MILVUS_TOKEN'] = str(MILVUS_TOKEN).strip()
os.environ['MILVUS_DB_NAME'] = str(MILVUS_DB_NAME).strip()
os.environ['EMBEDDING_API_KEY'] = str(EMBEDDING_API_KEY).strip()
os.environ['MINERU_OUTPUT_DIR'] = str(REPO_DIR / 'data' / 'mineru_output')
os.environ['HF_HOME'] = str(REPO_DIR / 'data' / 'hf_cache')
os.environ['HF_EMBED_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['HF_EMBED_BATCH_SIZE'] = '64' if torch.cuda.is_available() else '16'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# 如果你想在 notebook 里做一次临时试验，就用 EXTRA_ENV_OVERRIDES 覆盖。
for key, value in EXTRA_ENV_OVERRIDES.items():
    os.environ[str(key)] = str(value)

masked = {
    'env_template_path': str(env_template_path),
    'template_key_count': len(template_env),
    'MINERU_API_TOKEN_set': bool(os.environ.get('MINERU_API_TOKEN')),
    'MILVUS_URI_set': bool(os.environ.get('MILVUS_URI')),
    'MILVUS_TOKEN_set': bool(os.environ.get('MILVUS_TOKEN')),
    'MILVUS_DB_NAME': os.environ.get('MILVUS_DB_NAME', ''),
    'EMBEDDING_API_KEY_set': bool(os.environ.get('EMBEDDING_API_KEY')),
    'VECTOR_BACKEND': os.environ.get('VECTOR_BACKEND', ''),
    'RETRIEVAL_MODE': os.environ.get('RETRIEVAL_MODE', ''),
    'MINERU_OUTPUT_DIR': os.environ.get('MINERU_OUTPUT_DIR'),
    'HF_HOME': os.environ.get('HF_HOME'),
    'HF_EMBED_DEVICE': os.environ.get('HF_EMBED_DEVICE'),
    'HF_EMBED_BATCH_SIZE': os.environ.get('HF_EMBED_BATCH_SIZE'),
    'EXTRA_ENV_OVERRIDES': EXTRA_ENV_OVERRIDES,
}
(ARTIFACT_DIR / 'effective_ingest_env.json').write_text(
    json.dumps(masked, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(json.dumps(masked, ensure_ascii=False, indent=2))

if not os.environ['MINERU_API_TOKEN']:
    raise ValueError('MINERU_API_TOKEN is empty')
if not os.environ['MILVUS_URI']:
    raise ValueError('MILVUS_URI is empty')
if not os.environ['MILVUS_TOKEN']:
    raise ValueError('MILVUS_TOKEN is empty')
if os.environ.get('EMBEDDING_PROVIDER', '').strip().lower() == 'openai' and not os.environ['EMBEDDING_API_KEY']:
    raise ValueError('EMBEDDING_API_KEY is empty for openai-compatible embeddings')


In [ ]:
import subprocess
import sys
import time

# 直接调用主项目的 CLI 入口。
# 这里会实时打印 ingest 的标准输出，方便在 Colab 看进度。
cmd = [sys.executable, 'main.py', 'ingest', '--force']
print('running:', ' '.join(cmd))
start = time.time()
process = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
elapsed = time.time() - start
print(f'elapsed_sec={elapsed:.1f}')
if return_code != 0:
    raise RuntimeError(f'ingest failed with exit code {return_code}')


In [ ]:
import json
from zipfile import ZIP_DEFLATED, ZipFile

# 把本地 ask 真正需要的 data/cache，和后续增量入库很有用的
# data/mineru_output 一起打包带回本地。
cache_dir = REPO_DIR / 'data' / 'cache'
mineru_dir = REPO_DIR / 'data' / 'mineru_output'
bundle_zip = ARTIFACT_DIR / 'paper_rag_ingest_bundle.zip'

summary = {
    'cache_dir_exists': cache_dir.exists(),
    'mineru_output_exists': mineru_dir.exists(),
    'cache_files': sorted(str(p.relative_to(cache_dir)) for p in cache_dir.rglob('*') if p.is_file())[:200],
    'mineru_output_files': sum(1 for p in mineru_dir.rglob('*') if p.is_file()),
}
(ARTIFACT_DIR / 'ingest_artifact_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

with ZipFile(bundle_zip, 'w', compression=ZIP_DEFLATED) as zf:
    for root_dir in (cache_dir, mineru_dir):
        if not root_dir.exists():
            continue
        for file_path in root_dir.rglob('*'):
            if file_path.is_file():
                zf.write(file_path, arcname=str(file_path.relative_to(REPO_DIR)))
    zf.write(ARTIFACT_DIR / 'effective_ingest_env.json', arcname='effective_ingest_env.json')
    zf.write(ARTIFACT_DIR / 'ingest_artifact_summary.json', arcname='ingest_artifact_summary.json')

print('bundle_zip=', bundle_zip)
print(json.dumps(summary, ensure_ascii=False, indent=2))

if DOWNLOAD_AFTER_INGEST:
    from google.colab import files
    files.download(str(bundle_zip))


## 回本地怎么用

把 Colab 跑出来的这些内容放回本地项目同样的位置：

- `data/cache/`
- `data/mineru_output/`

然后保证你本地 `.env` 里的这些配置和 Colab 入库时保持一致：

- `MILVUS_URI`
- `MILVUS_TOKEN`
- `MILVUS_DB_NAME`
- `MILVUS_COLLECTION`
- `MILVUS_PAPERS_COLLECTION`
- `MILVUS_REFERENCES_COLLECTION`

满足这几点后，本地一般就可以直接：

```bash
python main.py ask "你的问题"
```

说明：
- `ask` 真正依赖的是本地 `data/cache` 和同一套云端 Milvus
- `data/mineru_output` 主要是为了你后续继续 `ingest` 时不重复做 PDF 解析
- 如果本地指向了另一套 Milvus collection，`ask` 会查到错库，或者直接提示先 `ingest`
